In [60]:
import pandas as pd
import pickle
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

In [61]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [62]:
rng = np.random.default_rng(73512)

In [63]:
with open("train.pkl", "rb") as f:
    raw_data = pickle.load(f)

print("Liczba próbek:", len(raw_data))

print("Przykładowa próbka:")
print(raw_data[0])

np.random.shuffle(raw_data)

Liczba próbek: 2939
Przykładowa próbka:
(array([ -1.,  -1.,  -1., ...,  78.,  40., 144.], shape=(4756,)), 0)


In [64]:
data = [x[0].astype(int) for x in raw_data]
targets = [int(x[1]) for x in raw_data]

print(pd.Series(targets).unique())
print(len(data))
print(data[0])
print(targets[0])


[0 3 1 4 2]
2939
[ -1  -1  -1  -1  67   0  47 180  47   0  92  80  92   0  47  47  73  12
 180 180  12 159 100 112 185 100 159 112  20   0  73 112 159 185 124 100
 100  92  20  20  88  33  20  20  33  33 113 113  56  33  88  20  20  44
  20  20  12   8  69 145 132 132  69 145  41  33 145 145   8 148 156 132
  12 189 110  76  12 157  44  20  20  74  77  92   6  92  45 124 100  37
 157  44  44  12  92  28 180 180   0  45  60   5  60  13  92 114 153  68
   7  80  85 160  41  80   2 153 153 121  68  68  13 180  73  47 180 119
   0   0   5  80   0   0  47  73  12 180  47 180  12 100   2 119 119  12
  88   0   0   0 114 119   2 119  78   0   0   0  92  93  93  93 142 124
 112 119 119  12   0   0  20  22 110  44  61 158 121  33  33 180 126  12
 140  29  60  89   1 128  42  43 140  24  81  36 180  35 112   2 119 119
   2 119  12   0   0   0   0  47 119  36 119  78   0   0   0  13  93  93
  52  93   5 119  12  12 100  53 119 119   2 180 112  35  73   0   5  52
   1   4   2 106   0   0 119 119  

In [65]:
from sklearn.model_selection import train_test_split


TRAIN_SPLIT = 0.7
X_train, X_val, y_train, y_val = train_test_split(
    data,
    targets,
    test_size=1 - TRAIN_SPLIT,
    stratify=targets
)

In [66]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

all_tokens = set()

for seq in X_train:
    all_tokens.update(seq)

all_tokens = sorted(list(all_tokens))

token2id = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1,
}

for token in all_tokens:
    token2id[token] = len(token2id)

id2token = {v: k for k, v in token2id.items()}

PAD_ID = token2id[PAD_TOKEN]
UNK_ID = token2id[UNK_TOKEN]

VOCAB_SIZE = len(token2id)

print("VOCAB_SIZE =", VOCAB_SIZE)

VOCAB_SIZE = 184


In [67]:
MAX_SEQ_LEN = 512

In [68]:
def encode_sequence(seq, max_len=MAX_SEQ_LEN):
    encoded = []

    for token in seq[:max_len]:
        encoded.append(token2id.get(token, UNK_ID))

    return torch.tensor(encoded, dtype=torch.long)

In [69]:
from torch.utils.data import Dataset

class VariableLenDataset(Dataset):
    def __init__(self, in_data, target):
        self.data = [(x, y) for x, y in zip(in_data, target)]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        in_data, target = self.data[idx]
        in_data = encode_sequence(in_data)
        return in_data, target

In [70]:
train_set = VariableLenDataset(X_train, y_train)
valid_set = VariableLenDataset(X_val, y_val)

print(len(train_set[0][0]))
print(len(train_set[1][0]))
print(train_set[0])


309
512
(tensor([  3,   3,   3,   8, 142,  44,   8,   3,   3,   8, 142,  44,  16, 111,
        111, 111, 111, 111, 111, 111, 111, 111,  16,  10,  87,  81,  81,  81,
        166, 166, 166,  81,  81,   5, 166,  59,  79, 135, 135,  66, 135, 135,
        135, 135, 135,   3,   3,   3, 109,   6,   4, 134,   3,   3, 109,   6,
          4,   8,  81, 142, 155,  44,  15, 123,  31,  11,  31,  31, 123,  15,
         15,  91,  91,  16,  44,  78, 123, 123, 155,   3,  83, 178, 118, 118,
        122,  78, 136, 182,  48,  42,  48,  48,  48, 122, 118, 118, 122,  78,
        136, 182,  48,  42,  48,  48,  48, 122,  15,  15, 122,  78, 101, 122,
        146,  16,  78,  16,  78,  78,  78,  78,  78,  78, 122,  47,   8, 153,
         47,  18,  11,  49,  50,  48,  49,  50,  48,  15, 178, 118,  15,  79,
         49,  50,  48,  49,  50,  48,  15, 110, 178, 110,  78,  31,   3, 115,
         16,  31,  31, 115, 115,   3,  15, 107, 107, 107, 107, 107,  47, 107,
        107, 107, 107, 107, 122, 182, 182,  48,  48, 18

In [71]:
num_classes = 5

class_counts = torch.bincount(torch.tensor(y_train), minlength=num_classes)

class_weights = 1.0 / class_counts.float()

# normalizacja opcjonalna
class_weights = class_weights / class_weights.sum() * num_classes

print(class_weights)

tensor([0.1954, 0.6675, 2.0644, 0.7215, 1.3512])


In [72]:
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

def pad_collate(batch, pad_value=PAD_ID):
    xx, yy = zip(*batch)
    x_lens = [len(x) for x in xx]

    xx_pad = pad_sequence(xx, batch_first=True, padding_value=pad_value)

    yy = torch.tensor(yy, dtype=torch.long)

    return xx_pad, yy, x_lens

In [73]:
BATCH_SIZE = 32
HIDDEN_SIZE = 32
EMBEDDING_SIZE = 16
OUT_SIZE = 5
NUM_LAYERS = 2
BIDIRECTIONAL = True
DROPOUT = 0.2
LR = 1e-3
TRAIN_EPOCHS = 10
CLIP_GRAD=False

In [74]:
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, collate_fn=pad_collate)
valid_loader = DataLoader(valid_set, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, collate_fn=pad_collate)

In [75]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size, num_layers, out_size, padding_idx=PAD_ID, bidirectional = False):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        if bidirectional:
            self.bidirectional = 2
        else:
            self.bidirectional = 1
        self.embedding = nn.Embedding(vocab_size, emb_size, padding_idx=padding_idx)
        self.lstm = nn.LSTM(input_size = emb_size, hidden_size = hidden_size, num_layers = num_layers, bidirectional=bidirectional, dropout=DROPOUT, batch_first=True)
        self.fc = nn.Linear(hidden_size*self.bidirectional, out_size)
        
    def init_hidden(self, batch_size):
        hidden = torch.zeros(self.num_layers*self.bidirectional , batch_size, self.hidden_size)
        state = torch.zeros(self.num_layers*self.bidirectional , batch_size, self.hidden_size)
        return hidden, state
    
    def forward(self, x, x_len, hidden=None):
        x = self.embedding(x)
        packed = nn.utils.rnn.pack_padded_sequence(x, x_len, batch_first=True, enforce_sorted=False)
        # if hidden is None:
        #     packed_out, (hn, cn) = self.lstm(packed)
        # else:
        packed_out, (hn, cn) = self.lstm(packed, hidden)
        if self.bidirectional == 1:
            out = hn[-1]
        else:
            forward_last = hn[-2]
            backward_last = hn[-1]
            out = torch.cat((forward_last, backward_last), dim=1)
                            
        return self.fc(out), (hn, cn)
    
model = LSTMClassifier(VOCAB_SIZE, EMBEDDING_SIZE, HIDDEN_SIZE, NUM_LAYERS, OUT_SIZE, padding_idx=PAD_ID, bidirectional=BIDIRECTIONAL).to(device)
model

LSTMClassifier(
  (embedding): Embedding(184, 16, padding_idx=0)
  (lstm): LSTM(16, 32, num_layers=2, batch_first=True, dropout=0.2, bidirectional=True)
  (fc): Linear(in_features=64, out_features=5, bias=True)
)

In [76]:
optimizer = torch.optim.Adam(model.parameters(), lr = LR)
loss_fun = nn.CrossEntropyLoss(weight=class_weights)

from tqdm import tqdm

model.train()
for epoch in tqdm(range(TRAIN_EPOCHS)):
    total_loss = 0
    for x, targets, x_len in train_loader:
        x = x.to(device)
        targets = targets.to(device)
        hidden, state = model.init_hidden(x.size(0))
        hidden, state = hidden.to(device), state.to(device) 
        preds, _ = model(x, x_len, (hidden,state))
        # preds, _ = model(x, x_len)
        loss = loss_fun(preds, targets)
        total_loss += loss.item()
        loss.backward()
        if CLIP_GRAD:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad() 
    if epoch % 1 == 0:
        print(f"Epoch: {epoch}, loss: {total_loss/len(train_loader):.3}")

 10%|█         | 1/10 [02:25<21:50, 145.65s/it]

Epoch: 0, loss: 1.56


 10%|█         | 1/10 [02:54<26:12, 174.75s/it]


KeyboardInterrupt: 

In [ ]:
model.eval()
preds= []
targets= []
total_loss = 0

with torch.no_grad():
    for x, target, x_len in valid_loader:
        x = x.to(device)
        target = target.to(device)
        hidden, state = model.init_hidden(x.shape[0])
        hidden, state = hidden.to(device), state.to(device)
        pred, _ = model(x, x_len, (hidden, state))
        loss = loss_fun(pred, target)
        total_loss += loss.item()

        preds.append(pred.cpu())
        targets.append(target.cpu())
print(f"Validation loss: {total_loss/len(valid_loader):.3}")
print(f"Accuracy: {(torch.argmax(torch.cat(preds),1).cpu()==torch.cat(targets)).sum().item()/len(torch.cat(targets)):.3}")

Validation loss: 0.719
Accuracy: 0.74


In [ ]:
print(torch.argmax(torch.cat(preds),1).cpu())

tensor([4, 1, 0, 3, 4, 2, 0, 0, 2, 0, 0, 1, 4, 3, 4, 3, 0, 0, 0, 4, 1, 0, 0, 1,
        0, 0, 0, 4, 4, 3, 0, 3, 3, 3, 1, 1, 0, 0, 0, 3, 1, 1, 0, 1, 0, 3, 0, 0,
        0, 0, 3, 0, 2, 0, 2, 1, 0, 0, 1, 3, 0, 3, 1, 0, 0, 0, 0, 0, 1, 0, 1, 3,
        0, 0, 2, 4, 1, 2, 3, 0, 1, 0, 3, 0, 1, 3, 0, 2, 3, 2, 4, 1, 0, 1, 4, 0,
        0, 0, 0, 0, 1, 0, 3, 0, 2, 0, 3, 1, 3, 0, 2, 0, 0, 3, 2, 3, 0, 0, 3, 3,
        4, 0, 3, 0, 0, 3, 0, 3, 0, 0, 3, 3, 0, 0, 3, 0, 0, 2, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 2, 0, 0, 2, 1, 0, 1, 3, 4, 0, 1, 1, 1, 0, 0, 0, 3, 0, 0,
        3, 3, 0, 0, 1, 0, 3, 1, 0, 0, 0, 1, 0, 1, 3, 0, 0, 3, 3, 0, 0, 1, 0, 0,
        0, 3, 0, 0, 0, 4, 4, 0, 0, 0, 4, 0, 0, 2, 0, 4, 1, 3, 0, 0, 0, 2, 3, 3,
        1, 0, 3, 3, 0, 1, 0, 0, 1, 0, 0, 2, 0, 2, 0, 4, 3, 0, 0, 4, 3, 0, 0, 1,
        0, 2, 1, 0, 0, 0, 0, 3, 2, 4, 0, 3, 0, 4, 0, 3, 1, 0, 0, 2, 0, 0, 0, 1,
        1, 4, 3, 2, 1, 1, 1, 3, 0, 3, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 4, 2, 1,
        1, 0, 1, 3, 0, 3, 0, 1, 4, 3, 0,

In [ ]:
print((torch.argmax(torch.cat(preds),1).cpu() == 0).sum())
print((torch.argmax(torch.cat(preds),1).cpu() != 0).sum())

tensor(449)
tensor(433)
